# Yaku: Entrenamiento de Modelos de Aprendizaje Automático para Riego de Lechuga en Suelo

Este notebook cubre las tres etapas del modelado de Machine Learning para predecir decisiones de riego en el cultivo de **Lechuga** utilizando telemetría de suelo y ambiente:

1. **Recopilación y Preprocesamiento del Conjunto de Datos (Limpieza)**
2. **Análisis Exploratorio de Datos (EDA)**
3. **Selección y Entrenamiento del Algoritmo junto con sus Métricas de Evaluación**

### Contexto de Sensores
Ambos entrenamientos (tomate y lechuga) se basan en los mismos datos físicos de sensores:
- Humedad del suelo y del ambiente.
- Temperatura del suelo y del ambiente.
- Etapa de crecimiento.

Para generar las etiquetas de entrenamiento, implementamos el perfil agronómico específico para la **lechuga** sobre los datos de sensores recopilados.

---


## 0. Preparación del Entorno
Instalamos y cargamos las librerías necesarias. Si ejecutas esto en Google Colab, te pedirá cargar el archivo del dataset de sensores (ej. `tomato irrigation dataset.csv`).


In [ ]:
# Carga de librerías esenciales
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import joblib

# Para interactuar con archivos en Google Colab
try:
    from google.colab import files
    in_colab = True
except ImportError:
    in_colab = False

# Estilo visual premium para gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Librerías importadas con éxito.")


## 1. Recopilación y Preprocesamiento del Conjunto de Datos (Limpieza)

En esta sección realizamos las siguientes operaciones de limpieza y transformación de los datos de sensores:
- Subida y lectura del dataset CSV.
- Limpieza de espacios en blanco en los nombres de las columnas.
- Renombrado de variables a un formato estándar en español utilizando coincidencia de subcadenas.
- Inferencia física de la temperatura del suelo.
- Mapeo categórico de las etapas de crecimiento a valores numéricos enteros.
- Eliminación de registros nulos en las características clave de entrada (`FEATURES`).


In [ ]:
def read_csv_with_fallback(source):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    last_error = None
    for encoding in encodings:
        try:
            if hasattr(source, "seek"):
                source.seek(0)
            df_loaded = pd.read_csv(source, encoding=encoding)
            print(f"CSV cargado usando encoding: {encoding}")
            return df_loaded
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

if in_colab:
    print("Por favor, sube el archivo de datos de telemetría de suelo (ej. 'lettuce_dataset.csv'):")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    df = read_csv_with_fallback(io.BytesIO(uploaded[file_name]))
else:
    import os
    default_path = './src/resources/ml_artifacts/dataset/lettuce_dataset.csv'
    if os.path.exists(default_path):
        df = read_csv_with_fallback(default_path)
        print(f"Cargado dataset local desde: {default_path}")
    else:
        print("No se detectó entorno Colab ni archivo local en la ruta por defecto. Por favor, especifica la ruta de tu archivo CSV:")
        path = input("Ruta del CSV: ")
        df = read_csv_with_fallback(path)

print(f"Dimensiones iniciales del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()


In [ ]:
# A. Limpiar espacios en blanco en nombres de columnas
df.columns = df.columns.str.strip()

# B. Renombrar columnas clave al español de forma robusta (evitando caracteres especiales)
rename_dict = {}
for col in df.columns:
    col_lower = col.lower()
    if "soil" in col_lower and ("moist" in col_lower or "humid" in col_lower):
        rename_dict[col] = "humedad_suelo"
    elif "temp" in col_lower:
        rename_dict[col] = "temperatura_ambiente"
    elif "humid" in col_lower:
        rename_dict[col] = "humedad_ambiente"
    elif "stage" in col_lower:
        rename_dict[col] = "etapa_original"
    elif "growth" in col_lower and "day" in col_lower:
        rename_dict[col] = "dias_crecimiento"
    elif "tds" in col_lower:
        rename_dict[col] = "tds_ppm"

df = df.rename(columns=rename_dict)

# C. Adaptaciones para lettuce_dataset.csv: no trae etapa ni humedad de suelo directamente.
if "etapa_original" not in df.columns and "dias_crecimiento" in df.columns:
    growth_days = pd.to_numeric(df["dias_crecimiento"], errors="coerce")
    df["etapa_original"] = pd.cut(
        growth_days,
        bins=[0, 10, 25, 40, float("inf")],
        labels=["initial stage", "development stage", "mid stage", "last stage"],
        include_lowest=True,
    ).astype(str)

if "humedad_suelo" not in df.columns and "tds_ppm" in df.columns:
    tds = pd.to_numeric(df["tds_ppm"], errors="coerce")
    tds_min = tds.min()
    tds_max = tds.max()
    if pd.notna(tds_min) and pd.notna(tds_max) and tds_max != tds_min:
        df["humedad_suelo"] = 520 - ((tds - tds_min) / (tds_max - tds_min) * 240)
    else:
        df["humedad_suelo"] = np.nan
    print("Aviso: lettuce_dataset.csv no contiene humedad de suelo; se usó TDS como proxy normalizado.")

# D. Cálculo físico de la temperatura del suelo (aproximación)
df["temperatura_suelo"] = df["temperatura_ambiente"] - 1.5

# E. Mapear las etapas de crecimiento categóricas a valores numéricos
STAGES = {
    "initial stage": 0,
    "development stage": 1,
    "mid stage": 2,
    "last stage": 3,
}
df["etapa_crecimiento"] = df["etapa_original"].astype(str).str.strip().str.lower().map(STAGES)

# F. Definición de características de entrada (FEATURES)
FEATURES = [
    "humedad_suelo",
    "humedad_ambiente",
    "temperatura_ambiente",
    "temperatura_suelo",
]

# G. Eliminar filas con valores faltantes en entradas del modelo y columnas para etiquetado
TRAINING_REQUIRED_COLUMNS = FEATURES + ["etapa_crecimiento"]
df_clean = df.dropna(subset=TRAINING_REQUIRED_COLUMNS).copy()

print(f"Variables de entrada del modelo ({len(FEATURES)}): {FEATURES}")

print(f"Dimensiones después de la limpieza: {df_clean.shape[0]} filas")
print("Mapeo de valores de etapa de crecimiento:")
print(df_clean['etapa_crecimiento'].value_counts(dropna=False))


## 2. Análisis Exploratorio de Datos (EDA)

Exploramos el dataset limpio para conocer el comportamiento de las variables físicas y del suelo antes de entrenar:
1. Estadísticas descriptivas de sensores de suelo y ambiente.
2. Gráficos de distribución de las variables meteorológicas y del suelo.
3. Matriz de correlación para evaluar relaciones de linealidad entre variables de suelo y clima.


In [ ]:
# 1. Resumen estadístico descriptivo
df_clean[FEATURES].describe()


In [ ]:
# 2. Gráficos de distribución
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Humedad del suelo
sns.histplot(df_clean['humedad_suelo'], kde=True, ax=axes[0, 0], color='#2b5c8f')
axes[0, 0].set_title('Distribución de la Humedad del Suelo (Lectura del Sensor)', fontsize=12)
axes[0, 0].set_xlabel('Humedad Suelo')
axes[0, 0].set_ylabel('Frecuencia')

# Humedad ambiente
sns.histplot(df_clean['humedad_ambiente'], kde=True, ax=axes[0, 1], color='#3a9e7a')
axes[0, 1].set_title('Distribución de la Humedad Ambiental (%)', fontsize=12)
axes[0, 1].set_xlabel('Humedad Ambiente (%)')
axes[0, 1].set_ylabel('Frecuencia')

# Temperatura ambiente
sns.histplot(df_clean['temperatura_ambiente'], kde=True, ax=axes[1, 0], color='#d95d39')
axes[1, 0].set_title('Distribución de la Temperatura Ambiental (°C)', fontsize=12)
axes[1, 0].set_xlabel('Temperatura (°C)')
axes[1, 0].set_ylabel('Frecuencia')

# Etapas de crecimiento
sns.countplot(x='etapa_original', data=df_clean, ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Distribución de Frecuencia por Etapa de Crecimiento del Cultivo', fontsize=12)
axes[1, 1].set_xlabel('Etapa')
axes[1, 1].set_ylabel('Conteo')
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
# 3. Visualizar pares de variables clave y su dispersion
plt.figure(figsize=(10, 8))
sns.pairplot(df_clean[['humedad_suelo', 'humedad_ambiente', 'temperatura_ambiente', 'etapa_original']], hue='etapa_original', palette='viridis')
plt.suptitle('Grafico de Dispersion por Pares y Etapa de Crecimiento', y=1.02)
plt.show()


In [ ]:
# 4. Matriz de Correlacion de Pearson
plt.figure(figsize=(8, 6))
corr_matrix = df_clean[FEATURES].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlación de Características de Suelo y Ambiente', fontsize=14)
plt.show()


## 3. Selección, Entrenamiento y Evaluación del Algoritmo

### 3.1. Generación de Etiquetas Basada en Perfil Agronómico de Lechuga
Implementamos la regla agronómica experta del backend Yaku para el cultivo de **Lechuga**. El límite de humedad y requerimiento climático se define como:
- **Perfil agronómico ideal (lechuga):** `CropProfile(humedad_suelo_max=360.0, humedad_ambiente_min=60.0, temperatura_ambiente_min=22.0, temperatura_suelo_min=21.0)`
- **Ajuste por etapa de crecimiento:** Días iniciales toleran menos humedad, días centrales requieren más. Ajuste de humedad crítica: `Etapa 0: -10, Etapa 1: 0, Etapa 2: +15, Etapa 3: -5`.
- **Decisión de Riego (`riego_requerido = 1`):** Suelo seco (`humedad_suelo` por debajo del umbral ajustado) + Demanda climática (temperatura alta o humedad ambiente baja).

### 3.2. División del Conjunto de Datos
Dividimos el conjunto de datos en **80% Entrenamiento** y **20% Prueba** utilizando una estratificación basada en `y` para balancear las clases en ambas particiones.


In [ ]:
# Perfiles de cultivo según requerimientos biológicos
class CropProfile:
    def __init__(self, humedad_suelo_max, humedad_ambiente_min, temperatura_ambiente_min, temperatura_suelo_min):
        self.humedad_suelo_max = humedad_suelo_max
        self.humedad_ambiente_min = humedad_ambiente_min
        self.temperatura_ambiente_min = temperatura_ambiente_min
        self.temperatura_suelo_min = temperatura_suelo_min

CROP_PROFILES = {
    "tomato": CropProfile(350.0, 65.0, 25.0, 24.0),
    "lettuce": CropProfile(360.0, 60.0, 22.0, 21.0),
}

def build_labels(data, profile, crop):
    if crop == "lettuce":
        stage_adjustment = data["etapa_crecimiento"].map({0: -10.0, 1: 0.0, 2: 15.0, 3: -5.0}).fillna(0.0)
        dry_soil = data["humedad_suelo"] < (profile.humedad_suelo_max + stage_adjustment)
        climatic_demand = (
            (data["temperatura_ambiente"] > profile.temperatura_ambiente_min)
            | (data["humedad_ambiente"] < profile.humedad_ambiente_min)
            | (data["temperatura_suelo"] > profile.temperatura_suelo_min)
        )
        return (dry_soil & climatic_demand).astype(int)
    else:
        stage_adjustment = data["etapa_crecimiento"].map({0: -20.0, 1: 0.0, 2: 25.0, 3: -10.0}).fillna(0.0)
        dry_soil = data["humedad_suelo"] < (profile.humedad_suelo_max + stage_adjustment)
        climatic_demand = (
            (data["temperatura_ambiente"] > profile.temperatura_ambiente_min)
            | (data["humedad_ambiente"] < profile.humedad_ambiente_min)
            | (data["temperatura_suelo"] > profile.temperatura_suelo_min)
        )
        return (dry_soil & climatic_demand).astype(int)

# 1. Seleccionar el cultivo para generar etiquetas (configurado para lechuga)
CROP_SELECTION = "lettuce"
print(f"Generando etiquetas de riego para: {CROP_SELECTION}")

df_clean["riego_requerido"] = build_labels(df_clean, CROP_PROFILES[CROP_SELECTION], CROP_SELECTION)

print("Distribución de la variable objetivo 'riego_requerido':")
print(df_clean["riego_requerido"].value_counts())
print(df_clean["riego_requerido"].value_counts(normalize=True))


In [ ]:
# 2. Preparar matrices X e y
X = df_clean[FEATURES]
y = df_clean["riego_requerido"]

# 3. Partición de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")


### 3.3. Algoritmo 1: Random Forest Classifier
Random Forest es un ensamble robusto de árboles de decisión. Usamos ponderación balanceada (`class_weight='balanced'`) para manejar cualquier desbalance de clases.


In [ ]:
# Inicializar y entrenar Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Entrenando modelo Random Forest...")
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print("¡Modelo entrenado exitosamente!")


### 3.4. Algoritmo 2: XGBoost Classifier
XGBoost es un algoritmo de Gradient Boosting extremo, altamente eficiente, que entrena árboles de forma secuencial minimizando los residuos de entrenamiento.


In [ ]:
# Inicializar y entrenar XGBoost
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

print("Entrenando modelo XGBoost...")
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
print("¡Modelo entrenado exitosamente!")


### 3.5. Métricas de Evaluación
Evaluaremos los modelos usando métricas clásicas para problemas de clasificación binaria:
- **Accuracy (Exactitud)**: Fracción de predicciones correctas.
- **Precision (Precisión)**: De las veces que el modelo predice 'Riego', ¿cuántas veces es correcto? (Evita desperdiciar agua).
- **Recall (Sensibilidad)**: De todos los momentos en que la lechuga realmente necesita agua, ¿cuántos detecta el modelo? (Evita estrés hídrico).
- **F1-Score**: Media armónica entre Precision y Recall.
- **Matriz de Confusión**: Distribución visual de Verdaderos Positivos, Verdaderos Negativos, Falsos Positivos y Falsos Negativos.
- **Importancia de Variables**: Evaluar qué sensores físicos de suelo y clima influyen más en la toma de decisión.


In [ ]:
def calculate_metrics(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"=== Métricas de {model_name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f} (evita falsas alarmas / desperdicio de agua)")
    print(f"Recall:    {rec:.4f} (evita deshidratación del cultivo)")
    print(f"F1-Score:  {f1:.4f}")
    print("\nReporte de clasificación detallado:")
    print(classification_report(y_true, y_pred, target_names=["No Riego", "Riego"], zero_division=0))
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1}

rf_metrics = calculate_metrics(y_test, rf_preds, "Random Forest")
print("\n" + "="*50 + "\n")
xgb_metrics = calculate_metrics(y_test, xgb_preds, "XGBoost")


In [ ]:
# Comparación visual de métricas
metrics_compare = pd.DataFrame({
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics
}).T

ax = metrics_compare.plot(kind="bar", figsize=(10, 6), ylim=(0.7, 1.05), colormap="viridis", edgecolor="black")
plt.title("Comparación del Rendimiento de Algoritmos (Lechuga)", fontsize=14, pad=15)
plt.ylabel("Puntuación", fontsize=12)
plt.xticks(rotation=0, fontsize=12)
plt.legend(loc="lower right", frameon=True, shadow=True)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.3f}", (p.get_x() * 1.005, p.get_height() * 1.01), fontsize=9)
plt.show()


In [ ]:
# Matrices de Confusión
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cm_rf = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Regar", "Regar"], yticklabels=["No Regar", "Regar"],
            cbar=False, annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title("Matriz de Confusión: Random Forest", fontsize=13, pad=10)
axes[0].set_ylabel("Valor Real (Sensor)", fontsize=11)
axes[0].set_xlabel("Valor Predicho (Modelo)", fontsize=11)

cm_xgb = confusion_matrix(y_test, xgb_preds)
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens", ax=axes[1],
            xticklabels=["No Regar", "Regar"], yticklabels=["No Regar", "Regar"],
            cbar=False, annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title("Matriz de Confusión: XGBoost", fontsize=13, pad=10)
axes[1].set_ylabel("Valor Real (Sensor)", fontsize=11)
axes[1].set_xlabel("Valor Predicho (Modelo)", fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Importancia de Variables en la Toma de Decisiones de Riego
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
rf_importances.plot(kind='barh', ax=axes[0], color='#3b629b', edgecolor='black')
axes[0].set_title('Importancia de las Características (Random Forest)', fontsize=13)
axes[0].set_xlabel('Importancia Relativa')

xgb_importances = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
xgb_importances.plot(kind='barh', ax=axes[1], color='#40a672', edgecolor='black')
axes[1].set_title('Importancia de las Características (XGBoost)', fontsize=13)
axes[1].set_xlabel('F-Score / Peso Relativo')

plt.tight_layout()
plt.show()


## 4. Exportación del Modelo
Una vez evaluados los modelos, exportamos los artefactos usando `joblib` para que la API de producción de Yaku pueda cargarlos dinámicamente para hacer inferencia en tiempo real a partir de los datos recibidos de los dispositivos ESP32.


In [ ]:
# Guardar modelos en formato binario
joblib.dump(rf_model, f"modelo_riego_{CROP_SELECTION}_rf.joblib")
joblib.dump(xgb_model, f"modelo_riego_{CROP_SELECTION}_xgb.joblib")
print("¡Modelos exportados correctamente como archivos .joblib!")

# Verificación: los .joblib deben clasificar usando exactamente los 4 parámetros de FEATURES
sample_input = X.iloc[[0]]
for model_name, model in {"Random Forest": rf_model, "XGBoost": xgb_model}.items():
    expected_features = getattr(model, "n_features_in_", len(FEATURES))
    if expected_features != len(FEATURES):
        raise ValueError(f"{model_name} espera {expected_features} variables, pero FEATURES tiene {len(FEATURES)}")
    sample_prediction = int(model.predict(sample_input)[0])
    print(f"{model_name}: OK, recibe {expected_features} variables y clasifica muestra como {sample_prediction}")

# Descargar los archivos del modelo en la máquina local si estás en Google Colab
if in_colab:
    import time
    try:
        print("Descargando el modelo Random Forest...")
        files.download(f"modelo_riego_{CROP_SELECTION}_rf.joblib")
        time.sleep(1.5)  # Espera para evitar que el navegador bloquee la segunda descarga
        print("Descargando el modelo XGBoost...")
        files.download(f"modelo_riego_{CROP_SELECTION}_xgb.joblib")
        print("Iniciando descargas automáticas en tu navegador...")
    except Exception as e:
        print("No se pudo descargar automáticamente, pero los archivos se guardaron en la pestaña de archivos laterales:", e)
